# Modelagem da Camada Gold: Dimensão Produto

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.services.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_dim_utils as modeling
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldDimProduto")

# Leitura das tabelas da camada Silver

In [ ]:
# Define caminhos das origens na Silver
silver_vendas_path = "s3a://silver/vendas"
silver_estoque_path = "s3a://silver/estoque"
silver_devolucoes_path = "s3a://silver/devolucoes"

# Lê os dados da Silver definindo como None caso a origem não exista
try:
    df_vendas = spark.read.parquet(silver_vendas_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Vendas não encontrada: {e}")
    df_vendas = None

try:
    df_estoque = spark.read.parquet(silver_estoque_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Estoque não encontrada: {e}")
    df_estoque = None

try:
    df_devolucoes = spark.read.parquet(silver_devolucoes_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Devoluções não encontrada: {e}")
    df_devolucoes = None

# Colunas de Produto em cada Origem

In [ ]:
# Mostra a pré-visualização das colunas caso a tabela não seja None
if df_vendas is not None:
    print("=== Colunas em Silver Vendas ===")
    display(df_vendas.select("produto_id", "produto", "categoria", "marca").limit(5).toPandas())

if df_estoque is not None:
    print("=== Colunas em Silver Estoque ===")
    display(df_estoque.select("produto_id", "produto", "categoria", "marca").limit(5).toPandas())

if df_devolucoes is not None:
    print("=== Colunas em Silver Devoluções ===")
    display(df_devolucoes.select("produto_id").limit(5).toPandas())

# Cria a Dimensão Produto (dim_produto)

In [ ]:
# Executa a lógica de modelagem unificada
df_dim_produto = modeling.create_dim_produto(df_vendas, df_estoque, df_devolucoes)

if df_dim_produto is not None:
    # Adiciona a data de carga
    df_dim_produto = df_dim_produto.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))
    
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de produtos únicos: {df_dim_produto.count()}")
    df_dim_produto.printSchema()
    display(df_dim_produto.limit(10).toPandas())
else:
    print("Nenhum produto foi processado (todas as tabelas Silver de origem estavam ausentes).")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)